# Phase 5 — Model Training, Evaluation, and Comparison

**Course:** DS605: Fundamentals of Machine Learning  
**Project:** Astronomical Object Classification Using Machine Learning  
**Dataset:** SDSS Stellar Classification Dataset (SDSS17)  
**Target Classes:** `GALAXY` (59.45%), `STAR` (21.59%), `QSO` (18.96%)  

### Abstract & Syllabus Alignment
In accordance with the **DS605 Course Syllabus**, this notebook explores the fundamental hypothesis classes and predictive frameworks taught throughout the semester:
- **Eager Learners:** Multinomial Logistic Regression with $L_2$ regularization.
- **Lazy Learners:** k-Nearest Neighbors (k-NN) with distance weighting.
- **Margin Classifiers:** Support Vector Machines (Linear SVM using Hinge Loss formulation).
- **Tree-Based Models:** Decision Tree Classifier (CART algorithm, Gini impurity criteria).
- **Ensemble Techniques:** Random Forest (Bagging + random feature subspaces), Histogram-Based Gradient Boosting (Sequential error minimization), and Soft Voting Ensembles.
- **Evaluation Methodology:** Leak-safe cross-validation with `StandardScaler` inside `Pipeline`, Stratified 5-Fold Cross-Validation, Class Imbalance handling via cost-sensitive weighting (`class_weight='balanced'`), and comprehensive multi-metric evaluation (Accuracy, Macro/Weighted F1, Confusion Matrices, and Multiclass One-vs-Rest ROC-AUC).

## 1. Imports, Environment Configuration, and Paths

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '1'
from pathlib import Path
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

# Aesthetic plot styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

current_dir = Path.cwd()
ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
DATA_FILE = ROOT / 'data' / 'processed' / 'sdss_features.csv'
MODELS_DIR = ROOT / 'models'
FIGURES_DIR = ROOT / 'figures' / 'evaluation'
REPORTS_DIR = ROOT / 'reports'

print(f"Project Root: {ROOT}")
print(f"Reading engineered dataset from: {DATA_FILE}")

Project Root: c:\Users\Akanksha\OneDrive\Desktop\ML_PROJECT2\ML Project\astronomy-ml-classification
Reading engineered dataset from: c:\Users\Akanksha\OneDrive\Desktop\ML_PROJECT2\ML Project\astronomy-ml-classification\data\processed\sdss_features.csv


## 2. Load Dataset & Train-Test Split

We partition the 99,999 observations into an **80% Training Set (79,999 samples)** and a **20% Held-Out Test Set (20,000 samples)**.
Stratification is enforced to guarantee identical class distributions across splits, preserving the natural class proportions: GALAXY (59.45%), STAR (21.59%), QSO (18.96%).

In [2]:
df = pd.read_csv(DATA_FILE)
CLASSES = ['GALAXY', 'STAR', 'QSO']

PHOTOMETRIC_FEATURES = [
    'u', 'g', 'r', 'i', 'z',
    'u_g', 'g_r', 'r_i', 'i_z',
    'u_r', 'u_z', 'g_z',
    'curvature_ugr', 'curvature_gri',
    'mag_mean', 'mag_std'
]

X = df[PHOTOMETRIC_FEATURES]
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Total dataset shape: {df.shape}")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print("\nClass proportions in Train set:")
print(y_train.value_counts(normalize=True).round(4))

Total dataset shape: (99999, 24)
Training set: 79999 samples
Test set:     20000 samples

Class proportions in Train set:
class
GALAXY    0.5945
STAR      0.2159
QSO       0.1896
Name: proportion, dtype: float64


## 3. Machine Learning Pipeline Setup (No Data Leakage)

To ensure strict scientific validity, every classifier is encapsulated inside a `Pipeline` containing `StandardScaler`. This guarantees that feature scaling parameters (mean $\mu$ and standard deviation $\sigma$) are fitted **only** on the training fold during cross-validation, preventing data snooping into validation or test folds.

In [3]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ]),
    'k-Nearest Neighbors': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=1))
    ]),
    'Linear SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearSVC(dual='auto', class_weight='balanced', C=1.0, max_iter=2000, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(criterion='gini', max_depth=12, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=1, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, class_weight='balanced', random_state=42))
    ])
}
print(f"Configured {len(models)} supervised learning architectures.")

Configured 6 supervised learning architectures.


## 4. Stratified 5-Fold Cross-Validation

We evaluate generalization performance using Stratified 5-Fold Cross Validation. We optimize for **Macro F1-Score** because Macro F1 treats all three classes equally, penalizing models that merely predict the majority class (GALAXY).

In [4]:
# Fast, representative stratified validation subset (10,000 samples)
cv_sample, _ = train_test_split(X_train, train_size=10000, stratify=y_train, random_state=42)
y_cv_sample = y_train.loc[cv_sample.index]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Stratified 5-Fold Cross-Validation Results (Macro F1) ===")
cv_results = {}
for name, pipe in models.items():
    t0 = time.time()
    scores = cross_val_score(pipe, cv_sample, y_cv_sample, cv=cv, scoring='f1_macro', n_jobs=1)
    duration = time.time() - t0
    cv_results[name] = (np.mean(scores), np.std(scores))
    print(f"{name:22s} | Mean Macro F1: {np.mean(scores):.4f} (+/- {np.std(scores):.4f}) | Time: {duration:4.1f}s")

=== Stratified 5-Fold Cross-Validation Results (Macro F1) ===
Logistic Regression    | Mean Macro F1: 0.6531 (+/- 0.0075) | Time:  0.4s
k-Nearest Neighbors    | Mean Macro F1: 0.8303 (+/- 0.0118) | Time:  1.2s
Linear SVM             | Mean Macro F1: 0.6710 (+/- 0.0115) | Time:  0.3s
Decision Tree          | Mean Macro F1: 0.7989 (+/- 0.0077) | Time:  0.7s
Random Forest          | Mean Macro F1: 0.8399 (+/- 0.0107) | Time: 10.3s
Gradient Boosting      | Mean Macro F1: 0.8440 (+/- 0.0084) | Time:  9.2s


## 5. Model Training & Evaluation on Held-Out Test Set

We now fit all models on the complete 79,999 training examples and evaluate on the 20,000 unobserved test instances.

In [5]:
test_results = []
trained_models = {}
y_test_pred_dict = {}
y_test_prob_dict = {}

lb = LabelBinarizer()
lb.fit(CLASSES)
y_test_bin = lb.transform(y_test)

# Efficient kNN training subsample for swift inference
knn_idx, _ = train_test_split(X_train.index, train_size=15000, stratify=y_train, random_state=42)

for name, pipe in models.items():
    t0 = time.time()
    if name == 'k-Nearest Neighbors':
        pipe.fit(X_train.loc[knn_idx], y_train.loc[knn_idx])
    else:
        pipe.fit(X_train, y_train)
    train_time = time.time() - t0
    trained_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_test_pred_dict[name] = y_pred
    
    clf = pipe.named_steps['clf']
    if hasattr(clf, 'predict_proba'):
        y_prob = pipe.predict_proba(X_test)
    elif hasattr(clf, 'decision_function'):
        df_scores = pipe.decision_function(X_test)
        exp_s = np.exp(df_scores - np.max(df_scores, axis=1, keepdims=True))
        y_prob = exp_s / np.sum(exp_s, axis=1, keepdims=True)
    else:
        y_prob = None
    y_test_prob_dict[name] = y_prob
    
    acc = accuracy_score(y_test, y_pred)
    f1_m = f1_score(y_test, y_pred, average='macro')
    f1_w = f1_score(y_test, y_pred, average='weighted')
    per_class = f1_score(y_test, y_pred, average=None, labels=CLASSES)
    auc_score = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='macro') if y_prob is not None else np.nan
    
    test_results.append({
        'Model': name,
        'CV_Macro_F1': cv_results[name][0],
        'Test_Accuracy': acc,
        'Macro_F1': f1_m,
        'Weighted_F1': f1_w,
        'F1_Galaxy': per_class[0],
        'F1_Star': per_class[1],
        'F1_QSO': per_class[2],
        'ROC_AUC_Macro': auc_score,
        'Train_Time_s': train_time
    })
    print(f"{name:20s} | Test Acc: {acc:.4f} | Macro F1: {f1_m:.4f} | Weighted F1: {f1_w:.4f}")

Logistic Regression  | Test Acc: 0.6786 | Macro F1: 0.6515 | Weighted F1: 0.6885
k-Nearest Neighbors  | Test Acc: 0.8673 | Macro F1: 0.8299 | Weighted F1: 0.8654
Linear SVM           | Test Acc: 0.7389 | Macro F1: 0.6691 | Weighted F1: 0.7306
Decision Tree        | Test Acc: 0.8579 | Macro F1: 0.8257 | Weighted F1: 0.8591
Random Forest        | Test Acc: 0.8820 | Macro F1: 0.8524 | Weighted F1: 0.8824
Gradient Boosting    | Test Acc: 0.8773 | Macro F1: 0.8481 | Weighted F1: 0.8784


## 6. Ensemble Learning: Soft Voting Classifier

We construct a Soft Voting Classifier combining Random Forest, Gradient Boosting, and Logistic Regression. In soft voting, predicted class probabilities are averaged: $P(y=c|x) = \frac{1}{M}\sum_{m=1}^M P_m(y=c|x)$.

In [6]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', trained_models['Random Forest']),
        ('hgb', trained_models['Gradient Boosting']),
        ('lr', trained_models['Logistic Regression'])
    ],
    voting='soft',
    n_jobs=1
)

t0 = time.time()
voting_clf.fit(X_train, y_train)
vote_train_time = time.time() - t0
y_pred_vote = voting_clf.predict(X_test)
y_prob_vote = voting_clf.predict_proba(X_test)

trained_models['Voting Ensemble'] = voting_clf
y_test_pred_dict['Voting Ensemble'] = y_pred_vote
y_test_prob_dict['Voting Ensemble'] = y_prob_vote

acc_v = accuracy_score(y_test, y_pred_vote)
f1_mv = f1_score(y_test, y_pred_vote, average='macro')
f1_wv = f1_score(y_test, y_pred_vote, average='weighted')
per_class_v = f1_score(y_test, y_pred_vote, average=None, labels=CLASSES)
auc_v = roc_auc_score(y_test_bin, y_prob_vote, multi_class='ovr', average='macro')

test_results.append({
    'Model': 'Voting Ensemble',
    'CV_Macro_F1': float(np.mean([cv_results['Random Forest'][0], cv_results['Gradient Boosting'][0]])),
    'Test_Accuracy': acc_v,
    'Macro_F1': f1_mv,
    'Weighted_F1': f1_wv,
    'F1_Galaxy': per_class_v[0],
    'F1_Star': per_class_v[1],
    'F1_QSO': per_class_v[2],
    'ROC_AUC_Macro': auc_v,
    'Train_Time_s': vote_train_time
})
print(f"Voting Ensemble Test Acc: {acc_v:.4f} | Macro F1: {f1_mv:.4f} | Weighted F1: {f1_wv:.4f}")

Voting Ensemble Test Acc: 0.8771 | Macro F1: 0.8475 | Weighted F1: 0.8778


## 7. Model Performance Comparison Table

In [7]:
comparison_df = pd.DataFrame(test_results).sort_values(by='Macro_F1', ascending=False).reset_index(drop=True)
print("\n================ SUMMARY BENCHMARK TABLE (PHOTOMETRY-ONLY) ================")
comparison_df[['Model', 'Test_Accuracy', 'Macro_F1', 'Weighted_F1', 'F1_Galaxy', 'F1_Star', 'F1_QSO', 'ROC_AUC_Macro']]


================ SUMMARY BENCHMARK TABLE (PHOTOMETRY-ONLY) ================


### Performance Bar Chart

In [8]:
fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(comparison_df))
w = 0.25

r1 = ax.bar(x - w, comparison_df['Test_Accuracy'], w, label='Accuracy', color='#2b5c8f')
r2 = ax.bar(x, comparison_df['Macro_F1'], w, label='Macro F1', color='#d97706')
r3 = ax.bar(x + w, comparison_df['Weighted_F1'], w, label='Weighted F1', color='#059669')

ax.set_title('Supervised Classification Models: Photometric Performance Comparison', fontsize=13, pad=12)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
ax.set_ylabel('Score (0.0 - 1.0)')
ax.set_ylim(0.4, 1.0)
ax.legend(frameon=True, loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_model_comparison_barchart.png', dpi=300)
plt.show()

## 8. Confusion Matrices & Error Analysis

Examining normalized confusion matrices reveals where models struggle. In pure photometry, the principal source of misclassification is between **Stars and Quasars**, because point-source high-redshift quasars can exhibit photometric colors similar to hot blue stars or white dwarfs without spectral line measurements.

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
eval_models = ['Random Forest', 'Gradient Boosting', 'Voting Ensemble', 'Decision Tree', 'Linear SVM', 'Logistic Regression']

for i, m_name in enumerate(eval_models):
    ax = axes[i]
    cm = confusion_matrix(y_test, y_test_pred_dict[m_name], labels=CLASSES)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES, cbar=False)
    ax.set_title(f"{m_name} (Acc: {accuracy_score(y_test, y_test_pred_dict[m_name]):.3f})")
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Normalized Confusion Matrices on Held-Out Test Set (N=20,000)', fontsize=14, y=0.99)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_confusion_matrices.png', dpi=300)
plt.show()

## 9. Multiclass One-vs-Rest ROC Curves

Receiver Operating Characteristic (ROC) curves evaluate the tradeoff between True Positive Rate (Sensitivity) and False Positive Rate ($1 - \text{Specificity}$) across varying decision thresholds.

In [10]:
rf_probs = y_test_prob_dict['Random Forest']
colors = {'GALAXY': '#2b5c8f', 'STAR': '#d97706', 'QSO': '#7c3aed'}

plt.figure(figsize=(8.5, 6))
for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], rf_probs[:, i])
    auc_val = roc_auc_score(y_test_bin[:, i], rf_probs[:, i])
    plt.plot(fpr, tpr, color=colors[cls], lw=2.5, label=f"{cls} (AUC = {auc_val:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance (AUC = 0.5000)')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('One-vs-Rest ROC Curves (Random Forest Classifier)', fontsize=13)
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_multiclass_roc_curves.png', dpi=300)
plt.show()

## 10. Gini Feature Importance (Random Forest)

Random Forest computes feature importance via **Mean Decrease in Impurity (MDI)**:
$$\text{MDI}(X_j) = \frac{1}{B} \sum_{b=1}^B \sum_{t \in T_b : v(t)=j} p(t) \Delta I(t)$$
where $\Delta I(t) = I(t) - \frac{N_{t_L}}{N_t} I(t_L) - \frac{N_{t_R}}{N_t} I(t_R)$ is the reduction in Gini impurity achieved by splitting on feature $X_j$.

In [11]:
rf_clf = trained_models['Random Forest'].named_steps['clf']
importances = pd.DataFrame({
    'Feature': PHOTOMETRIC_FEATURES,
    'Importance': rf_clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 5.5))
sns.barplot(data=importances, x='Importance', y='Feature', palette='viridis', hue='Feature', legend=False)
plt.title('Random Forest Feature Importance (Mean Decrease in Impurity)', fontsize=13)
plt.xlabel('Gini Importance')
plt.ylabel('Photometric Feature')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_feature_importance_rf.png', dpi=300)
plt.show()

## 11. Scientific Ablation Study: Impact of Spectroscopic Redshift

In astronomy surveys, redshift is determined through expensive **spectroscopy** (dispersing light into spectra to identify emission and absorption lines like $H_\alpha, H_\beta,$ and Ly-$\alpha$). 
Here, we conduct an ablation experiment: what happens if spectroscopic redshift is provided to the Random Forest model?

In [12]:
FEATURES_WITH_REDSHIFT = PHOTOMETRIC_FEATURES + ['redshift']
X_red = df[FEATURES_WITH_REDSHIFT]
y_red = df['class']

X_red_train, X_red_test, y_red_train, y_red_test = train_test_split(
    X_red, y_red, test_size=0.20, random_state=42, stratify=y_red
)

rf_red = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', n_jobs=1, random_state=42))
])
rf_red.fit(X_red_train, y_red_train)
y_pred_red = rf_red.predict(X_red_test)

acc_red = accuracy_score(y_red_test, y_pred_red)
f1_red = f1_score(y_red_test, y_pred_red, average='macro')

print(f"Random Forest WITHOUT Redshift (Photometry Only) : Accuracy = {comparison_df[comparison_df['Model']=='Random Forest']['Test_Accuracy'].values[0]:.4f}, Macro F1 = {comparison_df[comparison_df['Model']=='Random Forest']['Macro_F1'].values[0]:.4f}")
print(f"Random Forest WITH Spectroscopic Redshift         : Accuracy = {acc_red:.4f}, Macro F1 = {f1_red:.4f}")

Random Forest WITHOUT Redshift (Photometry Only) : Accuracy = 0.8820, Macro F1 = 0.8524
Random Forest WITH Spectroscopic Redshift         : Accuracy = 0.9756, Macro F1 = 0.9721


In [13]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
scenarios = ['Photometry Only (Realistic Survey)', 'Photometry + Spectroscopic Redshift']
acc_vals = [comparison_df[comparison_df['Model']=='Random Forest']['Test_Accuracy'].values[0], acc_red]
f1_vals = [comparison_df[comparison_df['Model']=='Random Forest']['Macro_F1'].values[0], f1_red]

x = np.arange(len(scenarios))
w = 0.3
b1 = ax.bar(x - w/2, acc_vals, w, label='Accuracy', color='#2b5c8f')
b2 = ax.bar(x + w/2, f1_vals, w, label='Macro F1', color='#d97706')
ax.set_ylabel('Score')
ax.set_title('Scientific Ablation Study: Value of Spectroscopic Redshift', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=10)
ax.set_ylim(0.6, 1.05)
ax.legend(frameon=True)

for bar in b1 + b2:
    h = bar.get_height()
    ax.annotate(f"{h:.4f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_photometry_vs_redshift_ablation.png', dpi=300)
plt.show()

## 12. Final Synthesis & Conclusions for Chalk Talk

1. **Champion Model:** **Random Forest** achieved the best overall performance with **88.20% Accuracy**, a **Macro F1-Score of 0.8524**, and a multiclass **ROC-AUC of 0.9653** on the held-out test set, closely followed by **HistGradientBoosting (87.74% Accuracy, 0.8481 Macro F1)** and the **Voting Ensemble (87.71% Accuracy, 0.8475 Macro F1)**.
2. **Linear vs Non-Linear Decision Boundaries:** Linear models (Logistic Regression at 67.86% and Linear SVM at 73.89%) significantly underperform non-linear tree ensembles. This demonstrates that astronomical SED color-color spaces possess complex non-linear decision manifolds.
3. **Physical Significance of Features:** The engineered color indices `u-g`, `g-r`, `r-i`, and `u-r` account for over 70% of feature importance in the Random Forest, validating that relative spectral energy slopes are the primary physical drivers of celestial taxonomy.
4. **The Redshift Dilemma:** Adding spectroscopic redshift increases classification accuracy to **97.56%**. However, because spectroscopic observations are 100 to 1,000 times more observationally expensive than imaging photometry, a practical wide-field telescope system must rely on photometric color classification, making our 88.2% baseline the true real-world imaging model.